# Getting started — data → features → model

A worked example of the full workflow on IndustryFlow: read your organization's sensor data,
engineer a few features, train a model, and register it so it appears on the **Models** page
(Configure → Models).

> **This notebook is read-only.** Run it to see the flow, then use **File → Save Notebook As…**
> to make your own editable copy.

Two things are already wired up for you and scoped to your organization — no credentials to
manage:

- **`sql_query`** reads your data over a **read-only** connection.
- **MLflow** tracks runs and registers models through the platform; you only ever see your own.

## 1. Read your data

Pull the last 7 days of readings from `sensor_measurements`. Queries are unqualified — they
resolve to your organization automatically.

In [ ]:
from industryflow import sql_query
import pandas as pd

df = sql_query(
    """
    SELECT time, sensor_id, value
    FROM sensor_measurements
    WHERE time > now() - interval '7 days'
    ORDER BY time
    """
)
print(f"{len(df):,} rows")
df.head()

In [ ]:
import numpy as np

# No data flowing yet? Synthesize a week of one sensor so the rest of the notebook still runs.
if df.empty:
    rng = pd.date_range(end=pd.Timestamp.utcnow(), periods=2016, freq="5min")
    val = 20 + 3 * np.sin(np.arange(len(rng)) / 48) + np.random.normal(0, 0.4, len(rng))
    df = pd.DataFrame({"time": rng, "sensor_id": "demo-sensor", "value": val})
    print("No live data — using a synthetic demo sensor.")

## 2. Engineer features

Take the busiest sensor, put it on a regular 15-minute grid, and build a few lag / rolling
features. The target is the next reading — a simple one-step-ahead forecast.

In [ ]:
top = df["sensor_id"].value_counts().index[0]
s = (
    df[df["sensor_id"] == top]
    .set_index("time")
    .sort_index()["value"]
    .astype(float)
    .resample("15min")
    .mean()
    .interpolate()
)

feat = pd.DataFrame({"value": s})
feat["lag_1"] = s.shift(1)
feat["lag_2"] = s.shift(2)
feat["roll_mean_4"] = s.rolling(4).mean()
feat["roll_std_4"] = s.rolling(4).std()
feat["hour"] = feat.index.hour
feat["target"] = s.shift(-1)  # next reading
feat = feat.dropna()
feat.head()

## 3. Train a model

A small random forest. We split without shuffling so the test set is genuinely "the future".

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

features = ["lag_1", "lag_2", "roll_mean_4", "roll_std_4", "hour"]
X, y = feat[features], feat["target"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, shuffle=False)

model = RandomForestRegressor(n_estimators=200, random_state=0)
model.fit(X_tr, y_tr)

pred = model.predict(X_te)
mae = float(mean_absolute_error(y_te, pred))
r2 = float(r2_score(y_te, pred))
print(f"MAE = {mae:.3f}   R2 = {r2:.3f}")

## 4. Track the run and register the model

`mlflow` is already pointed at the platform and scoped to your organization. Logging a run and
registering a model is the standard MLflow API — the platform handles keeping every
organization's experiments and models separate.

After this runs, open **Configure → Models** and you'll see `vpd-forecaster` with these metrics
and its version history.

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("getting-started")

with mlflow.start_run(run_name="rf-forecaster") as run:
    mlflow.log_params({"n_estimators": 200, "features": features})
    mlflow.log_metrics({"mae": mae, "r2": r2})
    mlflow.sklearn.log_model(
        model,
        name="model",
        registered_model_name="vpd-forecaster",
    )

print("Logged run:", run.info.run_id)
print("Registered model 'vpd-forecaster' — see it under Configure → Models.")

## What next

- Open **Configure → Models**, click `vpd-forecaster`, and add a **description** of what it does.
- **Save a copy** of this notebook (File → Save Notebook As…) and adapt the query, features, and
  model to your own equipment.
- Re-run the registration cell to publish a new **version** — they stack up in the model's history.

See the in-app **Help** guide (System → Help) for more on notebooks, data access, and models.